# Trabajo de contexto con scraping

### Primero debemos hacer loigin en una plataforma

- prueba del funcionamiento de la key 

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

MODEL = "gemini-2.0-flash"
openai = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta", api_key=api_key)

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "¿Cuánto son 2 + 2?"}]
)

print(response.choices[0].message.content)

2 + 2 = 4



### Primeros pasos

analizamos una pagina de coches de segunda mano para hacer scraping de dicha pagina 

In [3]:
import requests
from bs4 import BeautifulSoup

# A class to represent a Webpage
# Code from: https://github.com/ed-donner/llm_engineering 

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [4]:

novak_url = "https://www.autofesa.com/"
novak_web = Website(novak_url)
print(novak_web.title)
print(novak_web.text[350:400])

Concesionario segunda mano Madrid | Venta de coches ocasión
oches
Tablas GANVAM
Coches km0
Ofertas especiales 


In [5]:
def user_prompt_for(website):
   
    user_prompt = (
        f"Analiza el siguiente sitio web de coches de segunda mano y genera un resumen en formato Markdown.\n"
        f"### Título del sitio:\n{website.title}\n\n"
        "### Instrucciones:\n"
        "- Resume el propósito principal del sitio (por ejemplo: compraventa, anuncios, concesionario, comparador, etc.).\n"
        "- Destaca la información clave sobre los vehículos ofrecidos (marcas, modelos, precios, kilometraje, ubicación, etc.).\n"
        "- Si hay noticias, reseñas o secciones informativas sobre automóviles, inclúyelas brevemente.\n"
        "- Usa encabezados y listas en Markdown para organizar la información.\n"
        "- Evita texto irrelevante como menús, publicidad o pies de página.\n\n"
        "### Contenido del sitio:\n"
        f"{website.text.strip()}\n"
    )
    return user_prompt

In [6]:
system_prompt = "Eres un asistente que analiza el contenido de un sitio web \
                    y proporciona un resumen breve, ignorando el texto que podría estar relacionado con la navegación. \
                    No añades ningún comentario inicial ni final. \
                    Respondes en markdown. Respondes en español."


def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [7]:
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = MODEL,
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [8]:
summarize(novak_url)

'# Concesionario Autofesa: Venta de coches de segunda mano y ocasión en Madrid\n\nAutofesa es un concesionario online especializado en la compraventa de coches y motos de segunda mano y ocasión. Se posicionan como el concesionario número 1 en España, ofreciendo la posibilidad de comprar coches online con entrega a domicilio.\n\n## Información clave:\n\n*   **Vehículos ofrecidos:**\n    *   Coches de segunda mano, ocasión y Km 0.\n    *   Motos de segunda mano.\n    *   Furgonetas de ocasión.\n    *   Amplia variedad de marcas y modelos (Abarth, Audi, BMW, Citroen, Ford, Kia, Mercedes-Benz, Opel, Peugeot, Volvo, etc.).\n    *   Diversos tipos de vehículos: utilitarios, berlinas, familiares, todoterrenos, deportivos, monovolúmenes, eléctricos, híbridos, campers.\n\n*   **Servicios:**\n    *   Compraventa de coches y motos.\n    *   Tasación de vehículos (online y presencial).\n    *   Financiación.\n    *   Garantía en vehículos.\n    *   Taller propio para revisión y reparación.\n\n*   

In [9]:
from IPython.display import Markdown, display

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [10]:
display_summary(novak_url)

# Concesionario Autofesa: Venta de Coches de Segunda Mano y Ocasión en Madrid

## Propósito del Sitio

Autofesa es un concesionario online y físico dedicado a la compraventa de coches y motos de segunda mano y ocasión, ofreciendo también servicios de tasación y financiación.

## Información Clave de los Vehículos

*   **Variedad de Vehículos:** Amplia selección de coches de ocasión, motos, furgonetas y cámpers.
*   **Marcas:** Abarth, Alfa Romeo, Audi, BMW, Citroen, Ford, Kia, Mercedes-Benz, Opel, Peugeot, Renault, Seat, Skoda, Volkswagen, Volvo, entre otras.
*   **Modelos:** Diversos modelos disponibles dentro de cada marca.
*   **Kilometraje:** Opciones para filtrar por kilometraje máximo (desde menos de 10.000 km hasta más de 300.000 km).
*   **Precios:** Rango de precios variados, con filtros desde menos de 5.000 € hasta más de 60.000 €.
*   **Ubicación:** Concesionarios físicos en Collado Villalba, Leganés y Aravaca (Madrid).
*   **Tipos de Vehículos:** Utilitarios, berlinas, familiares, todoterrenos, pickups, deportivos, descapotables, monovolúmenes, eléctricos, híbridos, cámpers y furgonetas.
*   **Características Destacadas:**
    *   Más de 1.000 coches 100% revisados.
    *   Posibilidad de financiación.
    *   Aceptan coches como parte de pago.
    *   Entrega en toda la península (bajo condiciones).

## Servicios Ofrecidos

*   **Venta de coches de ocasión:** Amplio stock de vehículos revisados y garantizados.
*   **Compra de coches:** Tasación online y compra inmediata de vehículos.
*   **Financiación:** Opciones de financiación con bajas tasas de interés.
*   **Tasación GANVAM:** Servicio gratuito de tasación de vehículos.
*   **Taller:** Taller propio para revisión y reparación de vehículos con recambios originales.

## Secciones Informativas

*   **Blog:** Artículos sobre coches.
*   **Comparativas de coches:** Sección para comparar diferentes modelos.
*   **Opiniones de clientes:** Valoraciones y reseñas de clientes.
